# **1. Import Libraries & Load Data**

In [236]:
import pandas as pd
import re


In [237]:
datasets = {}

for year in range(2021, 2025):
    file = f"ttc-bus-delay-data-{year}.xlsx"
    sheets = pd.read_excel(file, sheet_name=None)

    datasets[year] = pd.concat(
        sheets.values(), ignore_index=True
    )

datasets[2025] = pd.read_csv(
    "TTC Bus Delay Data since 2025.csv",
    low_memory=False
)

# Display dataset dimensions
for year, df in datasets.items():
    print(f"{year}: {df.shape[0]} rows, {df.shape[1]} columns")

2021: 42269 rows, 13 columns
2022: 58707 rows, 10 columns
2023: 56207 rows, 10 columns
2024: 59643 rows, 10 columns
2025: 102257 rows, 11 columns


# **2. Data Inspection**

In [238]:
for year, df in datasets.items():
    print(f"\n========== YEAR {year} ==========")
    display(df.head(5))


========== YEAR 2021 ==========


,Date,Route,Time,Day,Location,Incident,Min Delay,Min Gap,Direction,Vehicle,Line,Bound,Unnamed: 10
0,2021-01-01,320,04:26,Friday,YONGE AND STEELES,Emergency Services,0,0,N,1553,NaN,NaN,NaN
1,2021-01-01,63,05:04,Friday,CANNIFF AND STRACHAN,Mechanical,10,0,N,8144,NaN,NaN,NaN
2,2021-01-01,42,05:34,Friday,WILSON GARAGE,Mechanical,24,48,N,8365,NaN,NaN,NaN
3,2021-01-01,320,05:44,Friday,YONGE AND STEELES,Emergency Services,15,30,N,8181,NaN,NaN,NaN
4,2021-01-01,35,06:21,Friday,JANE STATION,Vision,30,60,E,3186,NaN,NaN,NaN



========== YEAR 2022 ==========


,Date,Route,Time,Day,Location,Incident,Min Delay,Min Gap,Direction,Vehicle
0,2022-01-01,320,02:00,Saturday,YONGE AND DUNDAS,General Delay,0,0,NaN,8531
1,2022-01-01,325,02:00,Saturday,OVERLEA AND THORCLIFFE,Diversion,131,161,W,8658
2,2022-01-01,320,02:00,Saturday,YONGE AND STEELES,Operations - Operator,17,20,S,0
3,2022-01-01,320,02:07,Saturday,YONGE AND STEELES,Operations - Operator,4,11,S,0
4,2022-01-01,320,02:13,Saturday,YONGE AND STEELES,Operations - Operator,4,8,S,0



========== YEAR 2023 ==========


,Date,Route,Time,Day,Location,Incident,Min Delay,Min Gap,Direction,Vehicle
0,2023-01-01,91,02:30,Sunday,WOODBINE AND MORTIMER,Diversion,81,111,NaN,8772
1,2023-01-01,69,02:34,Sunday,WARDEN STATION,Security,22,44,S,8407
2,2023-01-01,35,03:06,Sunday,JANE STATION,Cleaning - Unsanitary,30,60,N,1051
3,2023-01-01,900,03:14,Sunday,KIPLING STATION,Security,17,17,NaN,3334
4,2023-01-01,85,03:43,Sunday,MEADOWALE LOOP,Security,1,1,NaN,1559



========== YEAR 2024 ==========


,Date,Route,Time,Day,Location,Incident,Min Delay,Min Gap,Direction,Vehicle
0,2024-01-01,89,02:08,Monday,KEELE AND GLENLAKE,Vision,10,20,N,7107
1,2024-01-01,39,02:30,Monday,FINCH STATION,General Delay,20,40,NaN,8914
2,2024-01-01,300,03:13,Monday,BLOOR AND MANNING,General Delay,0,0,NaN,8562
3,2024-01-01,65,03:23,Monday,PARLIAMENT AND BLOOR,Security,0,0,N,8574
4,2024-01-01,113,03:37,Monday,MAIN STATION,Security,0,0,NaN,8541



========== YEAR 2025 ==========


,_id,Date,Line,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Vehicle
0,1,2025-01-01T00:00:00,102 MARKHAM ROAD,02:15,Wednesday,WARDEN STATION,MFESA,20,40,N,3442
1,2,2025-01-01T00:00:00,65 PARLIAMENT,02:15,Wednesday,KIPLING STATION,MFUS,0,0,NaN,0
2,3,2025-01-01T00:00:00,64 MAIN,02:40,Wednesday,BROADVIEW STATION,MFUI,0,0,NaN,8546
3,4,2025-01-01T00:00:00,100 FLEMINGDON PARK,02:43,Wednesday,OVERLEA AND THORNCLIFF,MFSAN,17,32,N,8693
4,5,2025-01-01T00:00:00,34 EGLINTON EAST,03:05,Wednesday,EGLINTON AND DON MILLS,MFUI,20,40,W,8801


# **3. Data Integration & Standardization**

In [239]:
df_2021 = datasets[2021].copy()

# Combine Route and Line
df_2021['Route'] = df_2021['Route'].combine_first(
    df_2021['Line']
)

# Combine Direction and Bound
df_2021['Direction'] = df_2021['Direction'].combine_first(
    df_2021['Bound']
)

# Preserve the single additional note separately
extra_note = df_2021.loc[
    df_2021['Unnamed: 10'].notna(),
    ['Date', 'Route', 'Unnamed: 10']
].copy()

# Remove redundant columns
df_2021 = df_2021.drop(
    columns=['Line', 'Bound', 'Unnamed: 10']
)

# Update dataset
datasets[2021] = df_2021

print("2021 columns standardized successfully!")

print("\nUpdated columns:")
print(df_2021.columns.tolist())

print("\nDataset shape:", df_2021.shape)

print("\nPreserved additional note:")
display(extra_note)

2021 columns standardized successfully!

Updated columns:
['Date', 'Route', 'Time', 'Day', 'Location', 'Incident', 'Min Delay', 'Min Gap', 'Direction', 'Vehicle']

Dataset shape: (42269, 10)

Preserved additional note:


,Date,Route,Unnamed: 10
40063,2021-12-23,20,eglinton east expr


In [240]:
df_2025 = datasets[2025].copy()

df_2025['Date'] = pd.to_datetime(
    df_2025['Date'],
    errors='coerce'
)

# Rename columns to match previous years
df_2025 = df_2025.rename(columns={
    'Line': 'Route',
    'Station': 'Location',
    'Bound': 'Direction',
    'Code': 'Incident' 
})

# Preserve original record IDs separately
df_2025 = df_2025.drop(columns=['_id'])

# Update dataset
datasets[2025] = df_2025

print("2025 columns standardized successfully!")

print("\nUpdated columns:")
print(df_2025.columns.tolist())

print("\nDataset shape:", df_2025.shape)

print("\nFirst 5 records:")
display(df_2025.head())

2025 columns standardized successfully!

Updated columns:
['Date', 'Route', 'Time', 'Day', 'Location', 'Incident', 'Min Delay', 'Min Gap', 'Direction', 'Vehicle']

Dataset shape: (102257, 10)

First 5 records:


,Date,Route,Time,Day,Location,Incident,Min Delay,Min Gap,Direction,Vehicle
0,2025-01-01,102 MARKHAM ROAD,02:15,Wednesday,WARDEN STATION,MFESA,20,40,N,3442
1,2025-01-01,65 PARLIAMENT,02:15,Wednesday,KIPLING STATION,MFUS,0,0,NaN,0
2,2025-01-01,64 MAIN,02:40,Wednesday,BROADVIEW STATION,MFUI,0,0,NaN,8546
3,2025-01-01,100 FLEMINGDON PARK,02:43,Wednesday,OVERLEA AND THORNCLIFF,MFSAN,17,32,N,8693
4,2025-01-01,34 EGLINTON EAST,03:05,Wednesday,EGLINTON AND DON MILLS,MFUI,20,40,W,8801


In [241]:
# Keep records from 2025 and 2026 seperated
df_2026 = df_2025[
    df_2025['Date'].dt.year == 2026
].copy()

df_2025 = df_2025[
    df_2025['Date'].dt.year == 2025
].copy()

# Reset index
df_2025.reset_index(drop=True, inplace=True)
df_2026.reset_index(drop=True, inplace=True)

# Update dataset
datasets[2025] = df_2025
datasets[2026] = df_2026

print("2025 and 2026 datasets seperated successfully!")

2025 and 2026 datasets seperated successfully!


In [242]:
# Create a list to store datasets
all_datasets = []

for year, df in datasets.items():

    # Create a copy to preserve the original data
    temp_df = df.copy()

    # Add year column
    temp_df['Year'] = year

    # Add dataset to the list
    all_datasets.append(temp_df)

# Combine all datasets
df_combined = pd.concat(
    all_datasets,
    ignore_index=True,
    sort=False
)

# Display integration results
print("Data integration completed successfully!")

print("\nTotal rows:", df_combined.shape[0])
print("Total columns:", df_combined.shape[1])

print("\nRecords by year:")
print(df_combined['Year'].value_counts().sort_index())

print("\nCombined columns:")
print(df_combined.columns.tolist())

Data integration completed successfully!

Total rows: 319083
Total columns: 11

Records by year:
Year
2021    42269
2022    58707
2023    56207
2024    59643
2025    63029
2026    39228
Name: count, dtype: int64

Combined columns:
['Date', 'Route', 'Time', 'Day', 'Location', 'Incident', 'Min Delay', 'Min Gap', 'Direction', 'Vehicle', 'Year']


# **3. Data Type & Time Standardization**

In [243]:
df_not_cleaned = df_combined.copy()
df_cleaned = df_combined.copy()

print("Dataset Information:")
df_cleaned.info()

df_cleaned[["Day", "Location", "Incident", "Direction"]] = df_cleaned[["Day", "Location", "Incident", "Direction"]].astype('string')

print("\nData Types:")
print(df_cleaned.dtypes)

print("\nSample Records:")
display(df_cleaned.head())

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 319083 entries, 0 to 319082
Data columns (total 11 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   Date       319083 non-null  datetime64[ns]
 1   Route      316026 non-null  object        
 2   Time       319083 non-null  object        
 3   Day        319083 non-null  object        
 4   Location   319083 non-null  object        
 5   Incident   319083 non-null  object        
 6   Min Delay  319083 non-null  int64         
 7   Min Gap    319083 non-null  int64         
 8   Direction  260901 non-null  object        
 9   Vehicle    319083 non-null  int64         
 10  Year       319083 non-null  int64         
dtypes: datetime64[ns](1), int64(4), object(6)
memory usage: 26.8+ MB

Data Types:
Date         datetime64[ns]
Route                object
Time                 object
Day          string[python]
Location     string[python]
Incident     string[p

,Date,Route,Time,Day,Location,Incident,Min Delay,Min Gap,Direction,Vehicle,Year
0,2021-01-01,320,04:26,Friday,YONGE AND STEELES,Emergency Services,0,0,N,1553,2021
1,2021-01-01,63,05:04,Friday,CANNIFF AND STRACHAN,Mechanical,10,0,N,8144,2021
2,2021-01-01,42,05:34,Friday,WILSON GARAGE,Mechanical,24,48,N,8365,2021
3,2021-01-01,320,05:44,Friday,YONGE AND STEELES,Emergency Services,15,30,N,8181,2021
4,2021-01-01,35,06:21,Friday,JANE STATION,Vision,30,60,E,3186,2021


In [244]:
# Save backup
df_before_formatting = df_cleaned.copy()

# Function to standardize identifiers
def standardize_identifier(value):

    value = str(value).strip()

    # Remove .0 only from numeric identifiers
    if re.fullmatch(r'\d+\.0', value):
        value = value[:-2]

    return value

# Apply standardization
df_cleaned['Route'] = (
    df_cleaned['Route']
    .apply(standardize_identifier)
    .astype('string')
)

df_cleaned['Vehicle'] = (
    df_cleaned['Vehicle']
    .apply(standardize_identifier)
    .astype('string')
)

print("Route and Vehicle Formatting Completed!")

print("\nSample Routes:")
print(df_cleaned['Route'].head(10).tolist())

print("\nSample Vehicles:")
print(df_cleaned['Vehicle'].head(10).tolist())

print("\nDataset Shape:", df_cleaned.shape)

Route and Vehicle Formatting Completed!

Sample Routes:
['320', '63', '42', '320', '35', '29', '300', '300', '38', '29']

Sample Vehicles:
['1553', '8144', '8365', '8181', '3186', '9024', '8626', '8752', '3440', '9152']

Dataset Shape: (319083, 11)


In [245]:
# Convert time values for validation
time_values = pd.to_datetime(
    df_cleaned['Time'].astype(str),
    format='%H:%M',
    errors='coerce'
)

code_to_description = pd.read_csv('Code Descriptions.csv')

code_mapping = code_to_description.set_index('CODE')['DESCRIPTION'].to_dict()

df_cleaned['Incident'] = df_not_cleaned['Incident'].copy()
print(df_not_cleaned['Incident'])
print(df_cleaned['Incident'])

0         Emergency Services
1                 Mechanical
2                 Mechanical
3         Emergency Services
4                     Vision
                 ...        
319078                 EFCEL
319079                  MFSH
319080                  MFSH
319081                  MFUS
319082                  MFSH
Name: Incident, Length: 319083, dtype: object
0         Emergency Services
1                 Mechanical
2                 Mechanical
3         Emergency Services
4                     Vision
                 ...        
319078                 EFCEL
319079                  MFSH
319080                  MFSH
319081                  MFUS
319082                  MFSH
Name: Incident, Length: 319083, dtype: object


In [246]:
df_cleaned['Location'] = df_cleaned['Location'].str.title()

df_cleaned['Location'] = df_cleaned['Location'].replace({'@': "and", '&': "and"}, regex =False)

df_cleaned['Incident'] = df_cleaned['Incident'].replace(code_mapping)

df_cleaned['Incident'] = (df_cleaned['Incident'].astype('string')
    .str.strip().str.title().str.replace(r'\s+', ' ', regex=True)
)

incident_mapping = {
    'On Diversion': 'Diversion',
    'Collision - Ttc': 'Collision - TTC Involved',
    # 'Ttc Collision – Property Damage': 'TTC Collision – Property Damage',
    # 'Road Blocked - Non-Ttc Collision': 'Road Blocked - Non-TTC Collision',
    # 'Collision/Personal Injury - Ttc Not Involved': 'Collision/Personal Injury - TTC Not Involved',
    # 'Ttc Collision – Personal Injury': 'TTC Collision – Personal Injury',
    # 'Collision - Ttc Involved': 'Collision - TTC Involved'
}

df_cleaned['Incident'] = df_cleaned['Incident'].replace(incident_mapping)
df_cleaned['Incident'] = df_cleaned['Incident'].replace({r'Assault.*': 'Assault', 'Ttc': 'TTC'}, regex=True)

acronym_mapping = {
    'Esa': 'ESA',
    'Mfgd': 'MFGD',
    'Efcel': 'EFCEL',
    'Efsd': 'EFSD',
    'Sfl': 'SFL',
    'Effl': 'EFFL',
    'Efas': 'EFAS',
    'Mffl': 'MFFL',
    'Efvis': 'EFVIS',
    'Efti': 'EFTI',
    'Tfor': 'TFOR',
    'Eflt': 'EFLT',
    'Eftrm': 'EFTRM',
    'Mfbio': 'MFBIO',
    'Efws': 'EFWS',
    'Efwa': 'EFWA',
    'Mfie': 'MFIE',
    'Efce': 'EFCE',
    'Efse': 'EFSE',
    'Efvc': 'EFVC',
    'Tfdm': 'TFDM',
    'Tfav': 'TFAV',
    'Mfdi': 'MFDI',
    'Efex': 'EFEX',
    'Mfafr': 'MFAFR',
    'Effs': 'EFFS',
    'Efsp': 'EFSP',
    'Mfbd': 'MFBD',
    'Tfut': 'TFUT',
    'Mfse': 'MFSE',
    'Sfg': 'SFG',
    'Mfinf': 'MFINF',
    'Mfwr': 'MFWR',
    'Efpo': 'EFPO',
    'Efbk': 'EFBK',
}

df_cleaned['Incident'] = (
    df_cleaned['Incident']
    .astype('string')
    .str.replace('Â', '–', regex=False).str.strip()
)

df_cleaned['Incident'] = df_cleaned['Incident'].replace(acronym_mapping)

print(
    df_cleaned['Incident']
    .value_counts(dropna=False)
    .to_string()
)

Incident
Mechanical                                       65868
Operations - Operator                            49927
Other                                            22333
Diversion                                        19376
Security                                         15711
Collision - TTC Involved                         13995
General Delay                                    11901
Emergency Services                               10535
Utilized Off Route                                8989
Cleaning                                          8953
No Operator Available                             8742
Used As Shuttle Bus                               7730
TTC Collision – Property Damage                   6803
Cleaning - Unsanitary                             6598
Other – Switch Violation                          5593
Investigation                                     4359
Unable To Maintain Schedule                       4272
Vision                                            4097
D

In [247]:
original_routes = df_not_cleaned['Route'].copy()

df_cleaned['Route'] = (
    df_cleaned['Route'].astype('string')
    .str.extract(r'^(\d+)')
)

print(original_routes)

df_cleaned['Route'] =df_cleaned['Route'].combine_first(original_routes).astype('string')

print(df_cleaned['Route'])

0                   320
1                    63
2                    42
3                   320
4                    35
              ...      
319078    36 FINCH WEST
319079     63 OSSINGTON
319080          72 PAPE
319081          35 JANE
319082     106 SENTINEL
Name: Route, Length: 319083, dtype: object
0         320
1          63
2          42
3         320
4          35
         ... 
319078     36
319079     63
319080     72
319081     35
319082    106
Name: Route, Length: 319083, dtype: string


In [248]:
df_cleaned.drop(df_cleaned.loc[~df_cleaned['Direction'].isin(['N', 'S', 'E', 'W', 'B']) & df_cleaned['Direction'].notna()].index, inplace=True)
df_cleaned = df_cleaned.reset_index(drop=True)
print(df_cleaned.info())
print(df_not_cleaned['Direction'].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 319012 entries, 0 to 319011
Data columns (total 11 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   Date       319012 non-null  datetime64[ns]
 1   Route      315955 non-null  string        
 2   Time       319012 non-null  object        
 3   Day        319012 non-null  string        
 4   Location   319012 non-null  string        
 5   Incident   319012 non-null  string        
 6   Min Delay  319012 non-null  int64         
 7   Min Gap    319012 non-null  int64         
 8   Direction  260830 non-null  string        
 9   Vehicle    319012 non-null  string        
 10  Year       319012 non-null  int64         
dtypes: datetime64[ns](1), int64(3), object(1), string(6)
memory usage: 26.8+ MB
None
Direction
N    71187
S    67712
E    62781
W    58326
B      824
M        9
D        6
Q        4
8        4
J        4
`        4
G        4
7        4
2        4
A        3
\   

In [249]:
print(df_cleaned.head(5))

        Date Route   Time     Day              Location            Incident  \
0 2021-01-01   320  04:26  Friday     Yonge And Steeles  Emergency Services   
1 2021-01-01    63  05:04  Friday  Canniff And Strachan          Mechanical   
2 2021-01-01    42  05:34  Friday         Wilson Garage          Mechanical   
3 2021-01-01   320  05:44  Friday     Yonge And Steeles  Emergency Services   
4 2021-01-01    35  06:21  Friday          Jane Station              Vision   

   Min Delay  Min Gap Direction Vehicle  Year  
0          0        0         N    1553  2021  
1         10        0         N    8144  2021  
2         24       48         N    8365  2021  
3         15       30         N    8181  2021  
4         30       60         E    3186  2021  


# **4. Missing Values Handling**

In [250]:
# Count missing values in each column
missing_count = df_combined.isnull().sum()

# Calculate missing percentages
missing_percentage = (
    df_combined.isnull().mean() * 100
).round(2)

# Create missing values summary
missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing Percentage (%)': missing_percentage
})

# Sort by highest missing percentage
missing_summary = missing_summary.sort_values(
    by='Missing Percentage (%)',
    ascending=False
)

print("Missing Values Analysis")
print("Total records:", len(df_combined))

display(missing_summary)

Missing Values Analysis
Total records: 319083


,Missing Count,Missing Percentage (%)
Direction,58182,18.23
Route,3057,0.96
Date,0,0.00
Time,0,0.00
Day,0,0.00
Location,0,0.00
Incident,0,0.00
Min Delay,0,0.00
Min Gap,0,0.00
Vehicle,0,0.00


In [251]:
# Select important columns
columns_to_check = [
    'Route',
    'Direction',
    'Vehicle',
    'Min Delay',
    'Min Gap',
    'Location'
]

# Calculate missing values for each year
missing_by_year = (
    df_combined
    .groupby('Year')[columns_to_check]
    .apply(lambda x: x.isnull().sum())
)

print("Missing Values by Year:")
display(missing_by_year)

# Calculate missing percentages by year
missing_percentage_by_year = (
    df_combined
    .groupby('Year')[columns_to_check]
    .apply(lambda x: (x.isnull().mean() * 100).round(2))
)

print("\nMissing Percentages by Year:")
display(missing_percentage_by_year)

Missing Values by Year:


,Route,Direction,Vehicle,Min Delay,Min Gap,Location
Year,,,,,,
2021,240,7304,0,0,0,0
2022,384,10480,0,0,0,0
2023,570,9540,0,0,0,0
2024,622,9531,0,0,0,0
2025,757,11696,0,0,0,0
2026,484,9631,0,0,0,0



Missing Percentages by Year:


,Route,Direction,Vehicle,Min Delay,Min Gap,Location
Year,,,,,,
2021,0.57,17.28,0.0,0.0,0.0,0.0
2022,0.65,17.85,0.0,0.0,0.0,0.0
2023,1.01,16.97,0.0,0.0,0.0,0.0
2024,1.04,15.98,0.0,0.0,0.0,0.0
2025,1.20,18.56,0.0,0.0,0.0,0.0
2026,1.23,24.55,0.0,0.0,0.0,0.0


In [252]:
# Columns to clean
columns_with_missing_values = ['Route','Direction']

# Count missing values before cleaning
print("Missing values before cleaning:")
print(df_cleaned[columns_with_missing_values].isna().sum())

# Replace missing values with Unknown
for column in columns_with_missing_values:
    df_cleaned[column] = (
        df_cleaned[column]
        .astype('string')
        .fillna('Unknown')
    )

# Check results
print("\nMissing values after cleaning:")
print(df_cleaned[columns_with_missing_values].isna().sum())

print("\nDataset shape:", df_cleaned.shape)

Missing values before cleaning:
Route         3057
Direction    58182
dtype: int64

Missing values after cleaning:
Route        0
Direction    0
dtype: int64

Dataset shape: (319012, 11)


# **5. Duplicate Detection & Removal**

In [253]:
print("Total records:", len(df_cleaned))

# Count exact duplicate rows
duplicates = df_cleaned.duplicated().sum()

print("Total duplicate records:", duplicates)

# Display sample duplicates
print("\nSample duplicate records:")

display(
    df_cleaned[
        df_cleaned.duplicated(keep=False)
    ].head(10)
)

Total records: 319012
Total duplicate records: 1254

Sample duplicate records:


,Date,Route,Time,Day,Location,Incident,Min Delay,Min Gap,Direction,Vehicle,Year
158,2021-01-03,62,15:20,Sunday,Main Street Station,Cleaning,15,30,W,3138,2021
159,2021-01-03,62,15:20,Sunday,Main Street Station,Cleaning,15,30,W,3138,2021
212,2021-01-04,67,11:20,Monday,Victoria Park Station,Cleaning,15,30,N,8934,2021
213,2021-01-04,67,11:20,Monday,Victoria Park Station,Cleaning,15,30,N,8934,2021
231,2021-01-04,24,13:50,Monday,Victoria Park Station,Cleaning,10,20,N,8580,2021
232,2021-01-04,24,13:50,Monday,Victoria Park Station,Cleaning,10,20,N,8580,2021
268,2021-01-04,83,20:17,Monday,Commissioners And Lake,Mechanical,20,40,S,8502,2021
269,2021-01-04,83,20:17,Monday,Commissioners And Lake,Mechanical,20,40,S,8502,2021
479,2021-01-06,21,15:46,Wednesday,Scarborough Centre Sta,Cleaning,9,18,S,3508,2021
480,2021-01-06,21,15:46,Wednesday,Scarborough Centre Sta,Cleaning,9,18,S,3508,2021


In [254]:
# Identify exact duplicates
df_cleaned['Is Duplicate'] = df_cleaned.duplicated()

# Count duplicates by year
duplicates_by_year = (
    df_cleaned.groupby('Year')['Is Duplicate']
    .sum()
)

# Calculate duplicate percentages
duplicate_percentage = (
    df_cleaned.groupby('Year')['Is Duplicate']
    .mean() * 100
).round(2)

# Create summary table
duplicate_summary = pd.DataFrame({
    'Duplicate Count': duplicates_by_year,
    'Duplicate Percentage (%)': duplicate_percentage
})

print("Duplicate Analysis by Year:")
display(duplicate_summary)

# Remove temporary column
df_cleaned.drop(columns=['Is Duplicate'], inplace=True)

Duplicate Analysis by Year:


,Duplicate Count,Duplicate Percentage (%)
Year,,
2021,189,0.45
2022,367,0.63
2023,200,0.36
2024,198,0.33
2025,186,0.30
2026,114,0.29


In [255]:
# Count records before
rows_before = len(df_cleaned)

# Remove exact duplicates
df_cleaned = df_cleaned.drop_duplicates()

# Reset index
df_cleaned.reset_index(drop=True, inplace=True)

# Count records after
rows_after = len(df_cleaned)

print("Duplicate Removal Completed!")

print("\nRecords before:", rows_before)
print("Records after:", rows_after)
print("Duplicates removed:", rows_before - rows_after)

print("\nRemaining duplicates:",
      df_cleaned.duplicated().sum())

Duplicate Removal Completed!

Records before: 319012
Records after: 317758
Duplicates removed: 1254

Remaining duplicates: 0


In [256]:
print("FINAL DATA VALIDATION")
print("=" * 40)

# Dataset dimensions
print("\nTotal Records:", len(df_cleaned))
print("Total Columns:", len(df_cleaned.columns))

# Check duplicates (excluding the flag column)
print("\nDuplicate Records:",
      df_cleaned.duplicated().sum())

# Check important missing values
important_columns = [
    'Date', 'Time', 'Year',
    'Min Delay', 'Min Gap',
    'Route', 'Direction', 'Vehicle', 'Location'
]

print("\nMissing Values:")
print(df_cleaned[important_columns].isna().sum())

# Check negative delay values
print("\nNegative Delays:",
      (df_cleaned['Min Delay'] < 0).sum())

print("Negative Gaps:",
      (df_cleaned['Min Gap'] < 0).sum())

# Check year range
print("\nYears:",
      sorted(df_cleaned['Year'].unique()))

# Verify Date matches Year
year_mismatch = (
    df_cleaned['Date'].dt.year != df_cleaned['Year']
).sum()

print("Date-Year Mismatches:", year_mismatch)

# Final dataset shape
print("\nFinal Dataset Shape:", df_cleaned.shape)

FINAL DATA VALIDATION

Total Records: 317758
Total Columns: 11

Duplicate Records: 0

Missing Values:
Date         0
Time         0
Year         0
Min Delay    0
Min Gap      0
Route        0
Direction    0
Vehicle      0
Location     0
dtype: int64

Negative Delays: 0
Negative Gaps: 0

Years: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Date-Year Mismatches: 0

Final Dataset Shape: (317758, 11)


# **6. Export Cleaned Dataset**

In [257]:
# Save cleaned dataset as CSV
df_cleaned.to_csv('TTC_Bus_Delays_Cleaned_2020_2026.csv', index=False)

print("Dataset exported successfully!")

print("Final Shape:", df_cleaned.shape)

Dataset exported successfully!
Final Shape: (317758, 11)
